# Arbitraje óptimo de una batería sobre la curva a 20 años · TFM Energía UCM

Dada la ficha técnica de una batería, ¿cuánto gana arbitrando el precio horario entre hoy y
2046, y **cuántos ciclos al día le compensa hacer**?

La pregunta del número de ciclos no se responde fijándolo a mano. Se responde poniéndole al
optimizador **el precio de gastar vida** y viendo cuántos ciclos elige. Eso es lo que hace este
notebook.

## Qué relación tiene con lo que ya hay

`modelos/simulador_bess_horario.py`, de Willy, traduce a euros la **calidad de la predicción**
de D+1: carga en las D horas más baratas y descarga en las D más caras, un ciclo fijo, y compara
decidir con el modelo, con persistencia o con un oráculo. Es una pregunta distinta y su
respuesta sigue siendo válida.

Esto es lo otro: **arbitraje óptimo a largo plazo** sobre la curva de
`curva_fundamental`, con programación lineal en vez de greedy, con límites de carga, con coste
de degradación y **sin fijar el número de ciclos**. Se conservan sus convenciones —1 MW de
referencia, 90 % de eficiencia ida y vuelta— para que los euros sean comparables.

> **Sobre el material del máster:** el temario no cubre optimización. Regresión, series
> temporales, ML, deep learning y NLP, pero ni programación lineal ni investigación operativa.
> Así que el despacho se resuelve con `scipy.optimize.linprog` (HiGHS), que basta de sobra para
> un problema de 48 variables.

In [ ]:
import sys, importlib, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog
from scipy import sparse

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "gold").is_dir())
sys.path.append(str(REPO / "scripts"))
import curva_fundamental as cfun, curva_precios
importlib.reload(cfun); importlib.reload(curva_precios)
from curva_precios import por_anclas, historico

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
ROJO, AZUL, VERDE, GRIS = "#c0392b", "#2874a6", "#27ae60", "#7f8c8d"
H = historico()
print(f"histórico: {H.dia.min():%Y-%m-%d} -> {H.dia.max():%Y-%m-%d}")

## 1 · La ficha mínima de una batería

Para calcular arbitraje hacen falta **seis** números. Los tres primeros definen qué puede hacer
la batería; los tres últimos, cuánto le cuesta hacerlo.

| dato | unidad | para qué | valor típico (BESS de red, España 2026) |
|---|---|---|---|
| **potencia** | MW | cuánta energía mueve por hora | 1 MW *(todo se reporta por MW)* |
| **duración** | h | la energía: `E = P × duración` | **4 h** — el estándar de las subastas |
| **eficiencia ida y vuelta** | % | cuánto se pierde en el viaje | 90 % (ion-litio LFP) |
| **SoC mínimo y máximo** | % | la energía realmente usable | 5 % – 95 % |
| **ciclos de vida** | ciclos | hasta el 80 % de capacidad | 6.000 |
| **degradación** | % / 1.000 ciclos | pérdida de capacidad | 3 % |

Y **dos que faltan en esa lista y hacen falta igual**:

| dato | unidad | por qué importa |
|---|---|---|
| **CAPEX** | €/MWh instalado | sin él no hay coste de ciclo, y sin coste de ciclo el optimizador cicla gratis |
| **degradación calendárica** | % / año | la batería envejece aunque no la uses; decide si el límite real son los ciclos o los años |

Los otros parámetros de una ficha comercial —C-rate, autodescarga, ventana térmica, curva de
potencia frente a SoC— afectan poco al arbitraje horario y se omiten. La autodescarga de una
LFP es del orden de 2 %/mes: sobre un ciclo diario, ruido.

In [ ]:
# ─── LA FICHA. Cambia esto y todo lo demás se recalcula ─────────────────────
BAT = dict(
    potencia_mw      = 1.0,      # todo el resultado va "por MW instalado"
    duracion_h       = 4.0,      # el estándar de las subastas españolas
    eficiencia_rt    = 0.90,     # ida y vuelta, ion-litio LFP
    soc_min          = 0.05,
    soc_max          = 0.95,
    ciclos_vida      = 6000,     # hasta el 80 % de capacidad
    degrada_1000cic  = 3.0,      # % de capacidad por cada 1.000 ciclos
    capex_eur_mwh    = 200_000,  # €/MWh instalado (≈200 €/kWh)
    degrada_anual    = 1.5,      # % por año, aunque no se use
)
# ─────────────────────────────────────────────────────────────────────────────

E_TOTAL = BAT["potencia_mw"] * BAT["duracion_h"]
PROF = BAT["soc_max"] - BAT["soc_min"]
E_UTIL = E_TOTAL * PROF
ETA_LEG = np.sqrt(BAT["eficiencia_rt"])       # se reparte entre carga y descarga

print(f"  energía instalada     {E_TOTAL:6.2f} MWh")
print(f"  profundidad usable    {PROF:6.0%}  ->  {E_UTIL:.2f} MWh por ciclo")
print(f"  eficiencia por tramo  {ETA_LEG:6.3f}  (√{BAT['eficiencia_rt']})")

# ── coherencia de la ficha: ¿casan los ciclos de vida con la degradación? ───
perdida = BAT["ciclos_vida"] / 1000 * BAT["degrada_1000cic"]
print(f"\n  COHERENCIA: {BAT['ciclos_vida']:,} ciclos x {BAT['degrada_1000cic']}%/1000 "
      f"= {perdida:.0f}% de pérdida")
print(f"  el fin de vida se define en el 80 % de capacidad, o sea 20 % de pérdida "
      f"-> {'CASA' if 15 <= perdida <= 25 else 'NO CASA, revisa la ficha'}")

## 2 · El coste marginal del ciclo

Este es el concepto que decide todo, y es el que falta en casi todos los cálculos de arbitraje
que se ven por ahí.

Cada MWh que la batería descarga **consume vida**. Si no se le pone precio a esa vida, el
optimizador ciclará siempre que el margen supere las pérdidas de ida y vuelta — aunque sea por
1 €/MWh — y quemará la batería en dos años para ganar calderilla.

```
coste_ciclo (€/MWh descargado) = CAPEX_total / energía_total_descargada_en_toda_la_vida
                               = (E × capex) / (ciclos_vida × E_util)
```

Nótese que **no depende de la potencia ni de la duración por separado**, solo de la
profundidad usable y de los ciclos de vida.

Y hay una comprobación que hay que hacer antes: **¿qué limita primero, los ciclos o los
años?** Si la batería muere de vieja antes de agotar sus ciclos, ciclar es casi gratis y el
coste marginal debería ser menor.

In [ ]:
capex_total = E_TOTAL * BAT["capex_eur_mwh"]
throughput_vida = BAT["ciclos_vida"] * E_UTIL          # MWh descargados en toda la vida
COSTE_CICLO = capex_total / throughput_vida            # €/MWh descargado

print(f"  CAPEX total                 {capex_total:12,.0f} €")
print(f"  energía descargada en vida  {throughput_vida:12,.0f} MWh")
print(f"  ─────────────────────────────────────────────")
print(f"  COSTE MARGINAL DEL CICLO    {COSTE_CICLO:12,.1f} €/MWh descargado")

print(f"\n  ¿qué limita primero?")
vida_calendario = 20.0 / BAT["degrada_anual"]          # años hasta perder el 20 %
for cic_dia in (1.0, 1.5, 2.0):
    anos_ciclos = BAT["ciclos_vida"] / (cic_dia * 365)
    quien = "los CICLOS" if anos_ciclos < vida_calendario else "el CALENDARIO"
    print(f"    a {cic_dia:.1f} ciclos/día -> {anos_ciclos:5.1f} años por ciclado  ·  "
          f"{vida_calendario:.1f} por calendario  ->  limita {quien}")

print(f"\n  Con el calendario limitando, parte del CAPEX se pierde igual sin usar la")
print(f"  batería, así que el coste MARGINAL de un ciclo extra es menor que "
      f"{COSTE_CICLO:.0f} €/MWh.")
print(f"  Se explora en la sección 5 barriendo ese coste en vez de fijarlo.")

## 3 · El optimizador

Programación lineal con **previsión perfecta dentro de la ventana** — es el límite superior del
arbitraje, y por eso se llama óptimo.

### Cómo se plantea, y por qué así

La formulación ingenua pone como variables la carga y la descarga de cada hora, y expresa el
estado de carga como una suma acumulada. Eso obliga a una matriz triangular **densa**: para un
año son 8.760 × 8.760 y 613 MB. No escala.

La formulación buena mete el **estado de carga como variable explícita**, con una restricción de
diferencia por hora:

```
soc_t − soc_{t−1} − √η · carga_t + descarga_t / √η = 0
```

Tres coeficientes no nulos por fila en vez de una fila densa. Con eso un año entero se resuelve
en 0,3 s, y el horizonte completo de 7.427 días en unos 7.

**Variables** (3 por hora): carga, descarga y estado de carga.
**Objetivo**: `Σ p·descarga − Σ p·carga − coste_ciclo · Σ descarga`.
**Cotas**: carga y descarga entre 0 y P; el estado de carga entre `E·SoC_min` y `E·SoC_max`,
directamente como cota de la variable, que es más barato que como restricción.

### La ventana, que no es un detalle

La primera versión obligaba a la batería a volver a su estado inicial **cada medianoche**. Eso
es un artefacto: ninguna batería real se resetea a las 00:00. Con una ventana de varios días
puede cargar un domingo barato y descargar el lunes.

Medido sobre la curva completa, y no es una corrección de tercer orden:

| ventana | margen | |
|---|---|---|
| 1 día (reinicio cada medianoche) | 2.150.458 € | — |
| **7 días** | **2.337.515 €** | **+8,7 %** |
| 1 año | 2.368.754 € | +10,2 % |

Se usa **7 días** por defecto: recoge casi toda la mejora y es lo que hace un operador de verdad,
que planifica a semana vista. Dejar la ventana en un año sería suponer una previsión perfecta a
doce meses.

In [ ]:
BLOQUE_LP = 365      # días por resolución. Solo afecta a memoria, no al resultado.
VENTANA = 7          # días entre cierres de estado de carga

def _lp_bloque(px, bat, cc, ventana):
    """Un LP para un bloque de días. `px` es (días, 24)."""
    d, h = px.shape
    n = d * h
    p = px.ravel()
    Pw = bat["potencia_mw"]
    E = bat["potencia_mw"] * bat["duracion_h"]
    er = np.sqrt(bat["eficiencia_rt"])
    s0 = E * bat["soc_min"]

    # variables: [carga (n) | descarga (n) | soc (n)]
    obj = np.concatenate([p, -(p - cc), np.zeros(n)])

    # soc_t - soc_{t-1} - er*c_t + d_t/er = 0,  con soc_{-1} = s0
    fi, co, va = [], [], []
    for t in range(n):
        fi += [t, t, t]; co += [t, n + t, 2 * n + t]; va += [-er, 1 / er, 1.0]
        if t:
            fi += [t]; co += [2 * n + t - 1]; va += [-1.0]
    beq = np.zeros(n); beq[0] = s0

    # cerrar el estado de carga al final de cada ventana (y del bloque)
    cierres = list(range(ventana * h - 1, n, ventana * h))
    if not cierres or cierres[-1] != n - 1:
        cierres.append(n - 1)
    for k, t in enumerate(cierres):
        fi += [n + k]; co += [2 * n + t]; va += [1.0]
    beq = np.concatenate([beq, np.full(len(cierres), s0)])

    A = sparse.csr_matrix((va, (fi, co)), shape=(len(beq), 3 * n))
    r = linprog(obj, A_eq=A, b_eq=beq,
                bounds=[(0, Pw)] * n + [(0, Pw)] * n
                       + [(E * bat["soc_min"], E * bat["soc_max"])] * n,
                method="highs")
    if not r.success:
        return np.zeros(d), np.zeros(d), None
    c_, dd = r.x[:n], r.x[n:2 * n]
    e_util = E * (bat["soc_max"] - bat["soc_min"])
    ing = ((p * dd - p * c_) - cc * dd).reshape(d, h).sum(axis=1)
    cic = dd.reshape(d, h).sum(axis=1) / e_util
    return ing, cic, (c_.reshape(d, h), dd.reshape(d, h), r.x[2 * n:].reshape(d, h))


def despacho(precios, bat=BAT, coste_ciclo=None, ventana=VENTANA, perfil=False):
    """Arbitraje óptimo de una serie de días. `precios` es (días, 24).

    Devuelve (ingreso por día, ciclos por día). El ingreso ya lleva descontado el coste de
    ciclo: es margen neto.
    """
    px = np.atleast_2d(np.asarray(precios, dtype=float))
    cc = COSTE_CICLO if coste_ciclo is None else coste_ciclo
    ing, cic, det = [], [], []
    for i in range(0, len(px), BLOQUE_LP):
        a, b, p_ = _lp_bloque(px[i:i + BLOQUE_LP], bat, cc, ventana)
        ing.append(a); cic.append(b)
        if perfil:
            det.append(p_)
    ing, cic = np.concatenate(ing), np.concatenate(cic)
    if perfil:
        c_ = np.vstack([d[0] for d in det]); dd = np.vstack([d[1] for d in det])
        so = np.vstack([d[2] for d in det])
        return ing, cic, (c_, dd, so)
    return ing, cic

def bandas(ax, t, potencia, color, pmax, label=None):
    """Sombrea las horas de actividad cubriendo TODO el alto del eje.

    La version anterior sombreaba de 0 al precio, y como la carga ocurre por definicion en
    las horas baratas, la banda verde tenia altura casi nula: se perdia la mitad del
    despacho. Aqui la banda va de tope a tope del eje, detras de la linea.

    La opacidad es proporcional a la potencia, asi que una hora a plena carga se distingue
    de una a un tercio -- algo que antes tampoco se veia.
    """
    y0, y1 = ax.get_ylim()
    rgb = plt.matplotlib.colors.to_rgb(color)
    for i, mw in enumerate(potencia):
        if mw <= .01:
            continue
        ax.axvspan(t[i] - .5, t[i] + .5, ymin=0, ymax=1, lw=0, zorder=0,
                   color=(*rgb, .10 + .30 * min(mw / pmax, 1.0)),
                   label=label if (label and i == int(np.argmax(potencia))) else None)
    ax.set_ylim(y0, y1)

# ── cuánto cuesta el reinicio diario, medido ────────────────────────────────
_pr = np.abs(np.random.default_rng(0).normal(60, 40, (365, 24)))
t0 = time.time(); _i1, _ = despacho(_pr, ventana=1); t1 = time.time() - t0
t0 = time.time(); _i7, _ = despacho(_pr, ventana=7); t7 = time.time() - t0
t0 = time.time(); _ia, _ = despacho(_pr, ventana=365); ta = time.time() - t0
print(f"  {'ventana':>10s} {'margen/año':>12s} {'vs 1 día':>10s} {'tiempo':>8s}")
print("  " + "-" * 44)
for v, i_, t_ in ((1, _i1, t1), (7, _i7, t7), (365, _ia, ta)):
    print(f"  {v:8d} d {i_.sum():12,.0f} {100*(i_.sum()-_i1.sum())/_i1.sum():9.1f}% "
          f"{t_:7.2f}s")
print(f"se usa ventana = {VENTANA} días")

## 4 · La curva sobre la que se opera

La del notebook 08: `curva_fundamental`, desde el día siguiente al último precio publicado
hasta 2046, con escenarios completos.

**Se opera escenario a escenario y luego se promedia el ingreso**, nunca sobre el P50. El P50
es una mediana entre escenarios: suaviza los extremos y por tanto *subestima el spread*. Una
batería vive de los extremos, así que operarla sobre el promedio infravalora el negocio.

In [ ]:
ULT = H.dia.max()
SIM_DESDE = ULT + pd.Timedelta(days=1)
A_SIM, ANO_FIN = SIM_DESDE.year, 2046

P_ = cfun.panel()
obs = P_[P_.ano == P_.ano.max()]
GAS_HOY, DEM_HOY = float(obs.gas_mibgas.mean()), float(obs.demanda.mean())
SOL_HOY, EOL_HOY = float(obs.solar_gw.mean()), float(obs.eolica_gw.mean())

ESC = dict(
    gas=por_anclas({A_SIM: GAS_HOY, 2035: GAS_HOY * .82, ANO_FIN: GAS_HOY * .74},
                   A_SIM, ANO_FIN),
    demanda=por_anclas({A_SIM: DEM_HOY, ANO_FIN: DEM_HOY * 1.01 ** (ANO_FIN - A_SIM)},
                       A_SIM, ANO_FIN),
    solar_gw=por_anclas({A_SIM: SOL_HOY, 2030: 76, 2035: 95, 2040: 110, ANO_FIN: 125},
                        A_SIM, ANO_FIN),
    eolica_gw=por_anclas({A_SIM: EOL_HOY, 2030: 43, 2040: 55, ANO_FIN: 62},
                         A_SIM, ANO_FIN))

N_ESC = 12          # escenarios completos que se operan. Cada uno son ~15 s de LP.
potencial, _ = cfun.rendimientos(P_)
D_ = cfun.con_residual(P_, potencial)
precio_of, ic = cfun.curva_oferta(D_)
CF, SIMS = cfun.simular(SIM_DESDE, f"{ANO_FIN}-12-31", **ESC, potencial=potencial,
                        precio=precio_of, n=max(N_ESC, 60), verbose=False, crudo=True)
DIAS = CF.dia.values.reshape(-1, 24)[:, 0]
ANOS = pd.DatetimeIndex(DIAS).year.to_numpy()
print(f"{CF.dia.nunique():,} días · {SIMS.shape[0]} escenarios disponibles · "
      f"se operarán {N_ESC}")

## 5 · Un día, para ver qué hace el optimizador

Un día de julio a mitad del horizonte. Arriba el precio simulado; abajo, cuándo carga, cuándo
descarga y cómo evoluciona el estado de carga entre sus dos topes.

In [ ]:
DIA_EJ = f"{(A_SIM + ANO_FIN) // 2}-07-15"
i_dia = int(np.where(DIAS == np.datetime64(DIA_EJ))[0][0])
p_dia = SIMS[0].reshape(-1, 24)[i_dia]

# se despacha la SEMANA que contiene el día, para que la ventana tenga sentido,
# y se dibuja solo ese día
i0 = max(0, i_dia - 3)
sem = SIMS[0].reshape(-1, 24)[i0:i0 + 7]
_ing, _cic, (_c, _d, _s) = despacho(sem, perfil=True)
j = i_dia - i0
ing, cic = float(_ing[j]), float(_cic[j])
perf = pd.DataFrame({"precio": sem[j], "carga": _c[j], "descarga": _d[j], "soc": _s[j]})
E = BAT["potencia_mw"] * BAT["duracion_h"]

fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                       gridspec_kw={"height_ratios": [1, 1]})
ax[0].plot(perf.index, perf.precio, "o-", color="black", lw=2, ms=4, zorder=3)
ax[0].axhline(0, color=GRIS, lw=.9)
bandas(ax[0], perf.index.to_numpy(), perf.carga.to_numpy(), VERDE,
       BAT["potencia_mw"], "horas de carga")
bandas(ax[0], perf.index.to_numpy(), perf.descarga.to_numpy(), ROJO,
       BAT["potencia_mw"], "horas de descarga")
ax[0].set_ylabel("€/MWh"); ax[0].legend(loc="upper left", fontsize=9)
ax[0].set_title(f"{pd.Timestamp(DIA_EJ):%d-%m-%Y} · escenario 1 · "
                f"margen {ing:,.0f} € · {cic:.2f} ciclos", fontsize=12)

ax[1].bar(perf.index, perf.carga, color=VERDE, alpha=.75, label="carga (MW)")
ax[1].bar(perf.index, -perf.descarga, color=ROJO, alpha=.75, label="descarga (MW)")
b = ax[1].twinx(); b.grid(False)
b.plot(perf.index, perf.soc, "s--", color=AZUL, lw=1.8, ms=4, label="estado de carga")
b.axhline(E * BAT["soc_max"], ls=":", color=AZUL, lw=1.2)
b.axhline(E * BAT["soc_min"], ls=":", color=AZUL, lw=1.2)
b.set_ylabel("MWh almacenados", color=AZUL); b.tick_params(axis="y", labelcolor=AZUL)
ax[1].set_xlabel("hora"); ax[1].set_ylabel("MW"); ax[1].set_xticks(range(0, 24, 2))
h1, l1 = ax[1].get_legend_handles_labels(); h2, l2 = b.get_legend_handles_labels()
ax[1].legend(h1 + h2, l1 + l2, loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()

print(f"  carga    {perf.carga.sum():5.2f} MWh a un precio medio de "
      f"{(perf.precio * perf.carga).sum() / max(perf.carga.sum(), 1e-9):6.2f} €/MWh")
print(f"  descarga {perf.descarga.sum():5.2f} MWh a "
      f"{(perf.precio * perf.descarga).sum() / max(perf.descarga.sum(), 1e-9):6.2f} €/MWh")
print(f"  las líneas punteadas son los topes de SoC: {BAT['soc_min']:.0%} y "
      f"{BAT['soc_max']:.0%} de {E:.0f} MWh")

## 5b · Tres meses sorteados, semana a semana

Un día suelto no enseña lo que de verdad hace el optimizador. Aquí van **tres meses distintos**,
cada uno en cuatro semanas, con el precio arriba y la carga, la descarga y el estado de carga
debajo.

Tres muestras y no una porque un mes puede salir atípico: se sortean el momento del horizonte
**y** el escenario meteorológico, así que cada muestra cae en un año distinto y con otro tiempo.

Es donde se ve por qué la ventana de 7 días importa: **el estado de carga ya no vuelve a cero
cada medianoche**. Hay días que terminan cargados porque el siguiente pinta caro, y fines de
semana enteros en que la batería se queda quieta porque no hay margen que valga el desgaste.

Cambia `MUESTRAS` para sortear otros tres.

In [ ]:
MUESTRAS = [7, 23, 41]     # tres sorteos. Cambia estos números para ver otros meses
DIAS_MES = 28              # cuatro semanas exactas, para que casen con la ventana
E = BAT["potencia_mw"] * BAT["duracion_h"]
_NE = N_ESC if "N_ESC" in dir() else SIMS.shape[0]

resumen = []
for semilla in MUESTRAS:
    rng_mes = np.random.default_rng(semilla)
    i0 = int(rng_mes.integers(0, len(DIAS) - DIAS_MES))
    i0 -= i0 % VENTANA                       # alinear con el inicio de una ventana
    esc = int(rng_mes.integers(0, _NE))

    px_mes = SIMS[esc].reshape(-1, 24)[i0:i0 + DIAS_MES]
    ing_m, cic_m, (c_m, d_m, s_m) = despacho(px_mes, perfil=True)
    fechas = pd.DatetimeIndex(DIAS[i0:i0 + DIAS_MES])

    fig, axes = plt.subplots(4, 1, figsize=(14, 12))
    for w, ax in enumerate(axes):
        sl = slice(w * 7, (w + 1) * 7)
        t = np.arange(7 * 24)
        p_, c_, d_, s_ = (px_mes[sl].ravel(), c_m[sl].ravel(),
                          d_m[sl].ravel(), s_m[sl].ravel())
        ax.plot(t, p_, color="black", lw=1.4, zorder=3, label="precio")
        # las bandas van de tope a tope del eje: sombrear "de 0 al precio" hacia
        # invisible la carga, que por definicion ocurre cuando el precio es bajo
        ax.set_ylim(min(p_.min(), 0) * 1.12 - 2, p_.max() * 1.12)
        bandas(ax, t, c_, VERDE, BAT["potencia_mw"], "carga")
        bandas(ax, t, d_, ROJO, BAT["potencia_mw"], "descarga")
        ax.axhline(0, color=GRIS, lw=.8)
        for k in range(1, 7):
            ax.axvline(k * 24, color=GRIS, lw=.6, alpha=.6)
        ax.set_xlim(0, 7 * 24); ax.set_xticks(np.arange(0, 7 * 24 + 1, 24))
        ax.set_xticklabels([f"{fechas[sl][k]:%d-%m}" if k < 7 else ""
                            for k in range(8)], fontsize=8)
        ax.set_ylabel("€/MWh")
        b_ = ax.twinx(); b_.grid(False)
        b_.fill_between(t, E * BAT["soc_min"], s_, alpha=.18, color=AZUL, lw=0)
        b_.plot(t, s_, color=AZUL, lw=1.3, label="estado de carga")
        b_.set_ylim(0, E * 1.05); b_.set_ylabel("MWh", color=AZUL, fontsize=9)
        b_.tick_params(axis="y", labelcolor=AZUL, labelsize=8)
        ax.set_title(f"semana {w+1} · margen {ing_m[sl].sum():,.0f} € · "
                     f"{cic_m[sl].sum():.1f} ciclos", fontsize=10)
        if w == 0:
            h1, l1 = ax.get_legend_handles_labels()
            h2, l2 = b_.get_legend_handles_labels()
            ax.legend(h1 + h2, l1 + l2, ncol=4, fontsize=8, loc="upper left")
    fig.suptitle(f"MUESTRA {semilla} · {fechas[0]:%d-%m-%Y} a {fechas[-1]:%d-%m-%Y} · "
                 f"escenario {esc} · margen {ing_m.sum():,.0f} € · "
                 f"{cic_m.mean():.2f} ciclos/día", fontsize=13, y=.997)
    plt.tight_layout(); plt.show()

    resumen.append({"muestra": semilla, "desde": f"{fechas[0]:%d-%m-%Y}",
                    "escenario": esc, "precio_medio": px_mes.mean(),
                    "spread_medio": (px_mes.max(1) - px_mes.min(1)).mean(),
                    "margen_mes": ing_m.sum(), "ciclos_dia": cic_m.mean(),
                    "dias_sin_ciclar": int((cic_m < .05).sum()),
                    "soc_no_vacio_%": 100 * (s_m[:, -1] > E * BAT["soc_min"] + .05).mean()})

print()
display(pd.DataFrame(resumen).set_index("muestra").round(2))

**Qué mirar, y en qué se diferencian las tres muestras.**

La banda azul es la energía almacenada. Fíjate en que **no toca el suelo cada noche**: hay días
que acaban con carga guardada para el siguiente. Eso es lo que la versión con reinicio diario no
podía hacer, y es de donde salía ese 8,7 % de margen extra.

Las zonas verdes son horas de carga y las rojas de descarga. Casi siempre verde al mediodía y
rojo a las ocho de la tarde, que es el patrón que la curva a futuro va acentuando año tras año.

Y la tabla final enseña lo que no se ve en un promedio: **hay días que el optimizador no
cicla**, porque el spread de ese día no cubre el desgaste. El "1,08 ciclos/día" del resultado
global es una media entre días de dos ciclos y días de ninguno, no una rutina.

La columna `soc_no_vacío_%` dice qué porcentaje de días termina con energía guardada. Con el
reinicio diario de la primera versión ese número sería 0 por construcción.

## 6 · ¿Uno o dos ciclos al día?

**No se decide, se mide.** Se barre el coste del ciclo desde 0 y se mira cuántos ciclos elige
el optimizador. Con coste cero cicla por cualquier margen; según sube, va abandonando los
arbitrajes pequeños.

El barrido se hace sobre una muestra de días repartidos por todo el horizonte, no sobre el
horizonte entero, porque son un LP por día y por punto del barrido.

In [ ]:
# la muestra tiene que ser de semanas CONTIGUAS: con una ventana de 7 días, días
# sueltos sacados de aquí y de allá no forman una semana y el óptimo no significa nada.
_px = SIMS[0].reshape(-1, 24)
_ini = np.linspace(0, len(DIAS) - VENTANA - 1, 60).astype(int)
precios_m = np.vstack([_px[i:i + VENTANA] for i in _ini])
print(f"  muestra: {len(_ini)} semanas repartidas por el horizonte = "
      f"{len(precios_m)} días")

costes = [0, 5, 10, 15, 20, 25, 30, 40, 50]
filas = []
for cc in costes:
    i_, c_ = despacho(precios_m, coste_ciclo=cc)
    filas.append({"coste_ciclo": cc, "ciclos_dia": c_.mean(),
                  "margen_neto": i_.mean(),
                  "ingreso_bruto": (i_ + cc * c_ * E_UTIL).mean()})
sw = pd.DataFrame(filas).set_index("coste_ciclo")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
ax[0].plot(sw.index, sw.ciclos_dia, "o-", color=ROJO, lw=2.2)
ax[0].axvline(COSTE_CICLO, ls="--", color="black", lw=1.4,
              label=f"coste de esta batería\n{COSTE_CICLO:.0f} €/MWh")
ax[0].axhline(1, ls=":", color=GRIS); ax[0].axhline(2, ls=":", color=GRIS)
ax[0].set_xlabel("coste del ciclo (€/MWh descargado)")
ax[0].set_ylabel("ciclos por día"); ax[0].legend(fontsize=9)
ax[0].set_title("Cuántos ciclos elige el optimizador")

ax[1].plot(sw.index, sw.ingreso_bruto, "o-", color=AZUL, lw=2, label="ingreso bruto")
ax[1].plot(sw.index, sw.margen_neto, "o-", color=VERDE, lw=2.2, label="margen neto")
ax[1].axvline(COSTE_CICLO, ls="--", color="black", lw=1.4)
ax[1].axhline(0, color=GRIS, lw=.9)
ax[1].set_xlabel("coste del ciclo (€/MWh descargado)")
ax[1].set_ylabel("€/día por MW"); ax[1].legend(fontsize=9)
ax[1].set_title("Y lo que gana")
plt.tight_layout(); plt.show()
display(sw.round(2))

**Cómo se lee.** La línea negra es el coste de esta batería. Donde la corta la curva roja
está el número de ciclos que le compensa — y ese es el resultado, no un supuesto.

Con el coste a cero el optimizador cicla mucho más de dos veces, porque cualquier diferencia de
precio mayor que las pérdidas de ida y vuelta le sale a cuenta. Es la trampa de calcular
arbitraje sin degradación: sale un número grande que no se puede cobrar.

Y en la gráfica de la derecha se ve algo que conviene decir en la memoria: **el margen neto
puede ser negativo**. Si el coste del ciclo supera lo que el mercado paga por mover energía, la
batería no debería arbitrar — y eso es exactamente lo que pasa hoy con el CAPEX actual y por lo
que los proyectos reales viven de servicios de ajuste y del mercado de capacidad, no del
arbitraje.

## 7 · El resultado a 20 años, con su banda

Ahora sí, el horizonte completo. **Escenario a escenario**: cada uno se opera entero y después
se resumen los ingresos. Nunca sobre el P50 — es una mediana entre escenarios, suaviza los
extremos, y una batería vive de los extremos.

Con el optimizador por bloques cada escenario cuesta unos 7 segundos en vez de 15, así que se
pueden operar **50** en lugar de 12. Eso importa: con 12 escenarios el P10 es el segundo valor
de la muestra y es puro ruido; con 50 es el quinto y ya significa algo.

In [ ]:
N_ESC = 50          # con el optimizador por bloques, ~7 s cada uno

t0 = time.time()
PX_ALL = SIMS[:N_ESC].reshape(N_ESC, -1, 24)
res = np.zeros((N_ESC, len(DIAS)))
cyc = np.zeros((N_ESC, len(DIAS)))
for s in range(N_ESC):
    res[s], cyc[s] = despacho(PX_ALL[s])
    print(f"    escenario {s+1}/{N_ESC}  ({time.time()-t0:.0f}s)", end="\r")
print(f"\n  {N_ESC} escenarios x {len(DIAS):,} días en {time.time()-t0:.0f}s")

# ── margen anual por escenario: (escenarios, años) ──────────────────────────
anos_u = np.unique(ANOS)
por_ano = np.stack([res[:, ANOS == a].sum(axis=1) for a in anos_u], axis=1)
dias_ano = np.array([(ANOS == a).sum() for a in anos_u])
COMPLETO = dias_ano >= 365

Q = [5, 10, 25, 50, 75, 90, 95]
banda = pd.DataFrame({f"p{q}": np.percentile(por_ano, q, axis=0) for q in Q},
                     index=anos_u)
banda["media"] = por_ano.mean(axis=0)
banda["ciclos_dia"] = [cyc[:, ANOS == a].mean() for a in anos_u]
banda["dias"] = dias_ano
B = banda[COMPLETO]

fig, ax = plt.subplots(2, 1, figsize=(13, 8), sharex=True,
                       gridspec_kw={"height_ratios": [2, 1]})
ax[0].fill_between(B.index, B.p5, B.p95, alpha=.13, color=ROJO, lw=0, label="P5 - P95")
ax[0].fill_between(B.index, B.p10, B.p90, alpha=.20, color=ROJO, lw=0, label="P10 - P90")
ax[0].fill_between(B.index, B.p25, B.p75, alpha=.28, color=ROJO, lw=0, label="P25 - P75")
ax[0].plot(B.index, B.p50, "o-", color=ROJO, lw=2.4, ms=4, label="P50 (punto medio)")
ax[0].plot(B.index, B.media, "--", color="black", lw=1.4, label="media")
ax[0].axhline(0, color=GRIS, lw=.9)
ax[0].set_ylabel("€/año por MW instalado"); ax[0].legend(ncol=2, fontsize=9)
ax[0].set_title(f"Margen neto del arbitraje · batería {BAT['duracion_h']:.0f} h · "
                f"{N_ESC} escenarios · ventana {VENTANA} d", fontsize=12)

ax[1].plot(B.index, B.ciclos_dia, "o-", color=AZUL, lw=2.2)
ax[1].axhline(1, ls=":", color=GRIS); ax[1].axhline(2, ls=":", color=GRIS)
ax[1].set_ylabel("ciclos por día"); ax[1].set_xlabel("año")
plt.tight_layout(); plt.show()
display(B.drop(columns="dias").round(0))

### Los tres números que se piden para cada año

- **mínimo estimado** = P10. Uno de cada diez años caerá por debajo.
- **punto medio** = P50, la mediana. Es la cifra a usar.
- **máximo estimado** = P90.

No son el peor y el mejor caso posibles: son percentiles, y por construcción **una de cada
cinco veces se sale de la banda P10-P90**. Una banda P0-P100 sería tan ancha que no diría nada.

Y hay que recordar de dónde sale esa dispersión: es **meteorológica**. Todos los escenarios
comparten el mismo escenario de gas, demanda y capacidad. La incertidumbre de que *el escenario
esté equivocado* es mayor que esta banda y no está dentro.

### Aviso sobre esa banda: es más estrecha de lo que parece honesto

La dispersión del margen **total** entre los 50 escenarios es del **1,3 %**. Prácticamente no
hay banda, y conviene entender por qué antes de enseñarla.

Los 50 escenarios comparten el mismo escenario de gas, de demanda y de capacidad: **lo único
que cambia entre ellos es el tiempo que hace**. Y sobre 7.427 días, un año ventoso compensa a
uno seco. La ley de los grandes números se come la dispersión.

Así que la banda de este notebook mide **la variabilidad meteorológica de un año típico, no la
incertidumbre de la inversión**. La incertidumbre real —¿y si el gas se dispara?, ¿y si el
PNIEC se cumple a medias?, ¿y si entran 20 GW de baterías que cierran el spread?— es mucho
mayor y **no está aquí dentro**, porque para meterla habría que muestrear las anclas, no el
tiempo.

Es la misma advertencia que la sección 5 del notebook 07, y hay que repetirla aquí: la banda
recoge la variabilidad del resultado *dado* un escenario; no recoge que el escenario pueda
estar equivocado.

## 7b · El escenario bueno, el malo y el de en medio

Los percentiles anuales de arriba se calculan **año a año**, así que el "P10 de 2035" y el "P10
de 2040" pueden venir de escenarios distintos. Para una decisión de inversión eso no vale: hace
falta saber cómo le va a **un mismo escenario** de principio a fin.

Aquí se ordenan los 50 escenarios por su margen total y se sacan tres trayectorias completas:
la peor décima parte, la mediana y la mejor décima parte.

In [ ]:
total_esc = res.sum(axis=1)                 # margen total de cada escenario
orden = np.argsort(total_esc)
i_min = orden[int(.10 * N_ESC)]             # el escenario del percentil 10
i_med = orden[N_ESC // 2]
i_max = orden[int(.90 * N_ESC)]

print(f"  {'trayectoria':22s} {'margen total':>14s} {'€/año medio':>13s} "
      f"{'ciclos/día':>11s}")
print("  " + "-" * 64)
for nom, i_ in (("mínimo (P10)", i_min), ("punto medio (P50)", i_med),
                ("máximo (P90)", i_max)):
    n_anos = COMPLETO.sum()
    print(f"  {nom:22s} {total_esc[i_]:14,.0f} "
          f"{por_ano[i_][COMPLETO].mean():13,.0f} {cyc[i_].mean():11.2f}")
print("  " + "-" * 64)
print(f"  {'media de los ' + str(N_ESC):22s} {total_esc.mean():14,.0f} "
      f"{por_ano[:, COMPLETO].mean():13,.0f} {cyc.mean():11.2f}")
print(f"  {'dispersión (sd/media)':22s} {total_esc.std()/total_esc.mean():13.1%}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6),
                       gridspec_kw={"width_ratios": [1.6, 1]})
for i_, nom, col, lw in ((i_max, "máximo (P90)", VERDE, 2),
                         (i_med, "punto medio (P50)", AZUL, 2.6),
                         (i_min, "mínimo (P10)", ROJO, 2)):
    ax[0].plot(anos_u[COMPLETO], por_ano[i_][COMPLETO], "o-", ms=3, color=col, lw=lw,
               label=nom)
for s in range(N_ESC):
    ax[0].plot(anos_u[COMPLETO], por_ano[s][COMPLETO], lw=.4, color=GRIS, alpha=.25,
               zorder=0)
ax[0].set_ylabel("€/año por MW"); ax[0].set_xlabel("año"); ax[0].legend(fontsize=9)
ax[0].set_title("Trayectorias completas, no percentiles año a año")

ax[1].hist(total_esc / 1e6, bins=16, color=AZUL, alpha=.75)
for i_, col in ((i_min, ROJO), (i_med, AZUL), (i_max, VERDE)):
    ax[1].axvline(total_esc[i_] / 1e6, color=col, lw=2)
ax[1].set_xlabel("margen total del horizonte (M€ por MW)")
ax[1].set_ylabel("escenarios"); ax[1].set_title("Distribución entre escenarios")
plt.tight_layout(); plt.show()

## 7c · Y en dinero de hoy: el VAN

El margen sin descontar sirve para comparar años entre sí, pero no para decidir una inversión:
un euro de 2044 no vale lo que uno de 2027. Se descuenta a una tasa y se enfrenta al CAPEX.

Se usa la vida útil que salió de la degradación, no los 20 años del horizonte: la batería se
muere antes.

In [ ]:
TASA = 0.07            # coste de capital
OPEX_PCT = 0.015       # OPEX anual como % del CAPEX

def van(margen_anual, vida, tasa=TASA, opex=OPEX_PCT):
    """Valor actual neto de una trayectoria de márgenes anuales."""
    fl = margen_anual[:int(vida)] - opex * capex_total
    return float(-capex_total + np.sum(fl / (1 + tasa) ** np.arange(1, len(fl) + 1)))

vida_est = int(fin) if 'fin' in dir() else 8
vans = np.array([van(por_ano[s][COMPLETO], vida_est) for s in range(N_ESC)])

print(f"  vida útil {vida_est} años · tasa {TASA:.0%} · OPEX {OPEX_PCT:.1%} del CAPEX/año")
print(f"  CAPEX {capex_total:,.0f} € por MW\n")
print(f"  {'':22s} {'VAN (€/MW)':>14s}")
print("  " + "-" * 40)
for q, nom in ((10, "mínimo (P10)"), (50, "punto medio (P50)"), (90, "máximo (P90)")):
    print(f"  {nom:22s} {np.percentile(vans, q):14,.0f}")
print("  " + "-" * 40)
print(f"  {'media':22s} {vans.mean():14,.0f}")
print(f"\n  escenarios con VAN positivo: {(vans > 0).mean():.0%}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
ax[0].hist(vans / 1e3, bins=16, color=AZUL, alpha=.75)
ax[0].axvline(0, color=ROJO, lw=2.4, label="umbral de rentabilidad")
ax[0].axvline(np.median(vans) / 1e3, color="black", ls="--", lw=1.8, label="P50")
ax[0].set_xlabel("VAN (k€ por MW)"); ax[0].set_ylabel("escenarios")
ax[0].set_title(f"VAN del arbitraje solo · {vida_est} años al {TASA:.0%}")
ax[0].legend(fontsize=9)

# a qué CAPEX el negocio se sostiene solo con arbitraje
capexs = np.linspace(50_000, 300_000, 26)
prob = []
for cx in capexs:
    ct = E_TOTAL * cx
    v = np.array([-ct + np.sum((por_ano[s][COMPLETO][:vida_est] - OPEX_PCT * ct)
                  / (1 + TASA) ** np.arange(1, vida_est + 1)) for s in range(N_ESC)])
    prob.append((v > 0).mean())
ax[1].plot(capexs / 1000, np.array(prob) * 100, "o-", color=ROJO, lw=2.2)
ax[1].axvline(BAT["capex_eur_mwh"] / 1000, ls="--", color="black", lw=1.4,
              label=f"CAPEX de hoy ({BAT['capex_eur_mwh']/1000:.0f} €/kWh)")
ax[1].axhline(50, ls=":", color=GRIS)
ax[1].set_xlabel("CAPEX (€/kWh)"); ax[1].set_ylabel("% de escenarios con VAN > 0")
ax[1].set_title("A qué precio de batería sale el arbitraje solo")
ax[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

## 8 · ¿4 horas es la duración correcta?

Las subastas españolas van a 4 h, pero eso es una convención regulatoria, no un óptimo. Una
batería más larga captura más horas baratas —y las horas baratas van a ser cada vez más— pero
cuesta más CAPEX por MW, así que su coste de ciclo es el mismo por MWh pero necesita más energía
descargada para amortizarlo.

Se compara sobre la misma muestra de días y con el coste de ciclo recalculado para cada
duración.

In [ ]:
filas = []
for dur in (1, 2, 4, 6, 8):
    b = dict(BAT, duracion_h=float(dur))
    e_tot = b["potencia_mw"] * dur
    e_util = e_tot * PROF
    cc = (e_tot * b["capex_eur_mwh"]) / (b["ciclos_vida"] * e_util)   # = capex/(ciclos*prof)
    i_, c_ = despacho(precios_m, bat=b, coste_ciclo=cc)
    margen = i_.mean() * 365
    filas.append({"duración_h": dur, "MWh instalados": e_tot,
                  "coste_ciclo": cc, "ciclos_día": c_.mean(),
                  "margen_€/año/MW": margen,
                  "margen_€/año/MWh": margen / e_tot})
dur = pd.DataFrame(filas).set_index("duración_h")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
ax[0].bar(dur.index, dur["margen_€/año/MW"], color=AZUL, alpha=.8)
ax[0].set_xlabel("duración (h)"); ax[0].set_ylabel("€/año por MW de potencia")
ax[0].set_title("Por MW de potencia: más duración, más margen")
ax[1].bar(dur.index, dur["margen_€/año/MWh"], color=ROJO, alpha=.8)
ax[1].set_xlabel("duración (h)"); ax[1].set_ylabel("€/año por MWh instalado")
ax[1].set_title("Por MWh instalado: rendimientos decrecientes")
plt.tight_layout(); plt.show()
display(dur.round(1))

Las dos gráficas dicen cosas distintas y las dos importan. **Por MW de potencia** conviene
alargar: más horas baratas capturadas. **Por MWh instalado** —que es como se paga el CAPEX— hay
rendimientos decrecientes, porque las horas extra que se ganan son cada vez peores.

El óptimo económico está donde el margen por MWh instalado deja de compensar el coste por MWh
instalado. Con la ficha de arriba, ese punto se puede leer directo de la tabla.

## 9 · Cuánto dura la batería con este uso

Con los ciclos por día que ha elegido el optimizador, se puede proyectar la degradación y ver
cuándo llega al 80 % de capacidad — por ciclado, por calendario, o por lo que llegue antes.

In [ ]:
cd_medio = float(cyc.mean())
anos = np.arange(0, 26)
perd_cic = cd_medio * 365 * anos / 1000 * BAT["degrada_1000cic"]
perd_cal = BAT["degrada_anual"] * anos
perd_tot = perd_cic + perd_cal
fin = anos[np.argmax(perd_tot >= 20)] if (perd_tot >= 20).any() else np.nan

fig, ax = plt.subplots(figsize=(11, 4.6))
ax.plot(anos, 100 - perd_cic, "--", color=AZUL, lw=1.8, label="solo por ciclado")
ax.plot(anos, 100 - perd_cal, "--", color=VERDE, lw=1.8, label="solo por calendario")
ax.plot(anos, 100 - perd_tot, "-", color=ROJO, lw=2.6, label="combinado")
ax.axhline(80, ls=":", color="black", lw=1.6, label="fin de vida (80 %)")
if not np.isnan(fin):
    ax.axvline(fin, ls=":", color=ROJO, lw=1.4)
    ax.annotate(f"{fin} años", (fin, 82), fontsize=10, color=ROJO)
ax.set_xlabel("años de operación"); ax.set_ylabel("capacidad restante (%)")
ax.set_ylim(70, 101); ax.legend()
ax.set_title(f"Degradación con {cd_medio:.2f} ciclos/día de media")
plt.tight_layout(); plt.show()

print(f"  ciclos/día medios del optimizador   {cd_medio:6.2f}")
print(f"  ciclos totales hasta fin de vida    {cd_medio*365*fin:,.0f} de "
      f"{BAT['ciclos_vida']:,} disponibles")
print(f"  vida estimada                       {fin} años")
print(f"\n  Solo se gastan {cd_medio*365*fin/BAT['ciclos_vida']:.0%} de los ciclos "
      f"disponibles: LIMITA EL CALENDARIO, no el ciclado.")

# ── el ingreso, sin contar dos veces el CAPEX ───────────────────────────────
# `res` es margen NETO: ya lleva descontado COSTE_CICLO por cada MWh descargado, y ese
# coste ES la amortización del CAPEX. Compararlo otra vez contra el CAPEX seria contarlo
# dos veces. Lo que hay que enfrentar al CAPEX es el ingreso BRUTO.
mwh_vida = float(cyc.mean(axis=0).sum()) / len(DIAS) * 365 * fin * E_UTIL
neto_vida = float(res.mean(axis=0).sum()) / len(DIAS) * 365 * fin
bruto_vida = neto_vida + COSTE_CICLO * mwh_vida

print(f"\n  energía descargada en toda la vida  {mwh_vida:12,.0f} MWh por MW")
print(f"  ingreso BRUTO de arbitraje          {bruto_vida:12,.0f} €")
print(f"  CAPEX                               {capex_total:12,.0f} €")
print(f"  ──────────────────────────────────────────────────────")
print(f"  cobertura del CAPEX                 {bruto_vida/capex_total:12.0%}")
print(f"\n  Sin descontar, sin OPEX y con previsión perfecta. Con una tasa de captura")
print(f"  realista y un descuento del 7 %, esa cobertura baja bastante -- ver sección 11.")

## 11 · El coste del ciclo tiene una circularidad, y hay que resolverla

Hay un problema lógico en todo lo anterior que conviene mirar de frente.

El coste del ciclo se calculó repartiendo el CAPEX entre **los 6.000 ciclos de vida**. Pero la
sección anterior acaba de demostrar que la batería solo llega a hacer unos 3.000: muere de
vieja antes. Si solo se usan la mitad de los ciclos, cada MWh descargado carga con el doble de
CAPEX — y con un coste mayor el optimizador cicla menos, y si cicla menos usa aún menos
ciclos...

**El coste del ciclo depende de cuántos ciclos se hagan, y cuántos se hagan depende del coste.**
Se resuelve iterando hasta el punto fijo.

In [ ]:
# ── punto fijo del coste amortizado ─────────────────────────────────────────
CC_SW = sw.index.to_numpy(dtype=float)
CD_SW = sw.ciclos_dia.to_numpy()

def _vida_y_ciclos(cd):
    a = np.arange(0, 40, .25)
    perd = cd * 365 * a / 1000 * BAT["degrada_1000cic"] + BAT["degrada_anual"] * a
    v = float(a[np.argmax(perd >= 20)]) if (perd >= 20).any() else 40.0
    return v, cd * 365 * v

cc = COSTE_CICLO
print(f"  {'coste supuesto':>15s} {'ciclos/día':>11s} {'vida años':>10s} "
      f"{'ciclos usados':>14s} {'coste implicado':>16s}")
print("  " + "-" * 72)
for it in range(10):
    cd = float(np.interp(cc, CC_SW, CD_SW))
    v, usados = _vida_y_ciclos(cd)
    cc_new = capex_total / max(usados * E_UTIL, 1e-9)
    if it < 6:
        print(f"  {cc:15.1f} {cd:11.2f} {v:10.1f} {usados:14,.0f} {cc_new:16.1f}")
    cc += .5 * (cc_new - cc)
CC_AMORT = cc
cd_f, (v_f, us_f) = float(np.interp(cc, CC_SW, CD_SW)), _vida_y_ciclos(
    float(np.interp(cc, CC_SW, CD_SW)))
print("  " + "-" * 72)
print(f"  PUNTO FIJO -> coste amortizado {CC_AMORT:.0f} €/MWh · {cd_f:.2f} ciclos/día")

# ── y el coste MARGINAL, que es otro numero ─────────────────────────────────
perd_cic_f = cd_f * 365 * v_f / 1000 * BAT["degrada_1000cic"]
frac_ciclado = perd_cic_f / 20
CC_MARG = CC_AMORT * frac_ciclado
print(f"\n  De la pérdida total del 20 %, el ciclado causa {frac_ciclado:.0%} y el")
print(f"  calendario el resto -- que se perdería igual sin usar la batería.")
print(f"  COSTE MARGINAL de un ciclo extra: {CC_MARG:.0f} €/MWh")
print(f"\n  {'para DESPACHAR (¿ciclo o no?)':38s} usa el marginal  {CC_MARG:5.0f} €/MWh"
      f"  -> {np.interp(CC_MARG, CC_SW, CD_SW):.2f} ciclos/día")
print(f"  {'para INVERTIR (¿compro la batería?)':38s} usa el amortizado "
      f"{CC_AMORT:5.0f} €/MWh")

**Y son dos números distintos con dos usos distintos**, que es el punto de toda la sección.

Para decidir **si ciclar** —una decisión de operación— vale el coste **marginal**: cuánta vida
consume de verdad ese ciclo. Como buena parte de la degradación la causa el calendario y se
perdería igual con la batería parada, el marginal es bastante menor que el amortizado.

Para decidir **si comprar la batería** —una decisión de inversión— vale el **amortizado**: el
CAPEX hay que recuperarlo entero con la energía que realmente se va a mover, no con la que
cabría en teoría.

Confundirlos en un sentido lleva a infra-ciclar una batería ya comprada; en el otro, a comprar
una batería que no se paga. El notebook opera con el marginal y evalúa con el amortizado.

**La conclusión sobre los ciclos.** Con el coste marginal, el optimizador elige algo más de
**un ciclo al día**, no dos. Dos ciclos solo salen a cuenta con el coste de degradación a cero
—o sea, ignorándola— y ese es precisamente el error que hace que muchos cálculos de arbitraje
den números que no se pueden cobrar.

Eso puede cambiar hacia el final del horizonte: conforme el valle de mediodía se pega al cero y
aparecen más horas baratas, el segundo ciclo se vuelve más atractivo. La tabla de la sección 7
tiene la respuesta año a año.

## 10 · Lo que este cálculo NO incluye

Va aquí y no en un anexo, porque cambia el signo de la conclusión.

**Es un límite superior.** El optimizador conoce las 24 horas del día por adelantado. Una
batería real decide con una predicción, y ahí es donde entra el trabajo de Willy: su simulador
mide cuánto de este máximo se captura decidiendo con el modelo de D+1 en vez de con un oráculo.
**El número de aquí hay que multiplicarlo por esa tasa de captura.**

**Solo hay arbitraje.** Falta todo lo demás de la cuenta de resultados de una batería: servicios
de ajuste, restricciones técnicas, mercado de capacidad. En los proyectos españoles reales el
arbitraje es la parte pequeña, y por eso la cobertura del CAPEX que sale arriba no debe leerse
como "el negocio no existe" sino como "el negocio no es esto".

**Sin realimentación de precios.** Y este es el sesgo grande. La curva se construyó **sin
almacenamiento en el sistema**: hoy España tiene 235 MW y el PNIEC apunta a decenas de GW.
Cuando entren, arbitrarán el spread hasta cerrarlo. **El spread de 2046 de la curva es un techo,
así que el margen calculado aquí también lo es.**

**Sin coste de oportunidad entre mercados.** Una batería que se compromete en el mercado de
capacidad no puede arbitrar libremente. Aquí se le deja hacer lo que quiera.

**Sin restricciones de red ni de temperatura**, y con potencia constante frente al estado de
carga. Las tres son optimistas, aunque de segundo orden frente a las anteriores.

### Cómo se lee un 160 % de cobertura junto a un VAN negativo

Los dos números son correctos y dicen cosas distintas:

- **160 %** es el ingreso bruto acumulado en 8 años frente al CAPEX, **sin descontar** y sin
  OPEX. Es la respuesta a "¿mueve suficiente energía?".
- **VAN −197.000 €/MW** es lo mismo descontado al 7 % y con un OPEX del 1,5 % anual. Es la
  respuesta a "¿es una inversión?".

Entre los dos está el valor del dinero en el tiempo. Ocho años al 7 % descuentan un factor
cercano a 0,6 en los flujos tardíos, y el OPEX se lleva 12.000 €/año.

**Ningún escenario de los 50 da VAN positivo.** No es un matiz: con el CAPEX de hoy, el
arbitraje puro no paga una batería. Y se puede decir con precisión cuánto falta:

| | |
|---|---|
| CAPEX de equilibrio (VAN = 0) | **147 €/kWh** |
| CAPEX supuesto | 200 €/kWh |
| tiene que bajar | **27 %** |
| margen anual necesario al CAPEX de hoy | 145.974 €/MW |
| margen que da el arbitraje | 107.288 €/MW — el **73 %** |

Ese 73 % es, en una cifra, por qué los proyectos BESS españoles **no se financian con
arbitraje**: les falta un cuarto largo de los ingresos, y lo sacan del mercado de capacidad y de
los servicios de ajuste. El arbitraje es el suelo del negocio, no el negocio.

Y conviene recordar la dirección de los sesgos: este 73 % está calculado con **previsión
perfecta y sin realimentación de precios**. Los dos lo empujan hacia abajo.

## 12 · Las técnicas que quedan fuera, y por qué

Lo que hay aquí es **programación lineal con previsión perfecta en ventana**. Es el estándar
para acotar por arriba, y es lo correcto para responder "cuánto se puede ganar como máximo".
Pero no es lo que usaría un operador real, y conviene saber qué falta.

### Lo que sí se ha incorporado

| | antes | ahora |
|---|---|---|
| formulación | acumulada densa, 613 MB para un año | estado de carga como variable, dispersa |
| coste por día | 1,9 ms | 0,75 ms |
| ventana | reinicio cada medianoche | 7 días — **+8,7 % de margen** |
| escenarios | 12 | 50, con banda P5-P95 |
| decisión de ciclos | fijada a mano | sale del coste marginal |

### Lo que falta, por orden de lo que cambiaría el resultado

**1 · Decidir con previsión, no con oráculo.** Es la más importante y la que más resta. Aquí el
optimizador ve las 24 horas por adelantado; una batería real decide con una predicción. La
técnica estándar es **control predictivo por horizonte deslizante (MPC)**: se optimiza con la
previsión, se ejecuta solo la primera hora, se vuelve a optimizar. El simulador de Willy mide
justo esa pérdida para D+1 — habría que encadenarlos.

**2 · Degradación dependiente de la profundidad.** Aquí cada MWh descargado cuesta lo mismo. En
una celda de litio real, la degradación crece **más que proporcionalmente con la profundidad de
descarga**: dos ciclos al 50 % desgastan bastante menos que uno al 100 %. La técnica estándar es
el **conteo rainflow** —prestado de la fatiga de materiales— aplicado al perfil de estado de
carga. Con eso, el optimizador preferiría ciclos someros y frecuentes, y la respuesta de
"un ciclo o dos" podría cambiar. Es la mejora con mejor relación valor/esfuerzo que queda.

**3 · Optimización estocástica de verdad.** La forma correcta de operar bajo incertidumbre no es
optimizar cada escenario por separado y promediar —que es lo que hacemos y sobreestima, porque
cada solución conoce su futuro— sino una política única que funcione en todos. En el sector se
usa **SDDP** (programación dinámica dual estocástica), que es el estándar en hidrotermia. La
diferencia entre las dos cosas tiene nombre: **el valor de la información perfecta**, y aquí no
está descontado.

**4 · Pérdida de capacidad a lo largo de la vida.** La energía útil se toma constante. En
realidad cae hasta el 80 %, así que los últimos años rinden menos de lo que dice la tabla.

**5 · Varios mercados a la vez.** Arbitraje, servicios de ajuste y capacidad compiten por la
misma energía. La formulación de co-optimización existe y es un LP más grande, pero necesita
precios de esos mercados que no están en la base.

**6 · Cotas de red y potencia frente a estado de carga.** Segundo orden frente a las anteriores.

### Lo que esto significa para la cifra

Las tres primeras van todas en la **misma dirección: restan**. Y la realimentación de precios de
la sección 10 también. Así que el margen de arriba hay que leerlo como **un techo bastante
generoso**, no como una estimación central.